In [ ]:
# =========================
# INSTALAÇÃO
# =========================
!pip install scikit-learn pandas joblib nltk unidecode spacy
!python -m spacy download pt_core_news_md -q

# =========================
# IMPORTS
# =========================
import pandas as pd
import re
import joblib
import numpy as np
from unidecode import unidecode
import spacy

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

stopwords_pt = stopwords.words('portuguese')

nlp = spacy.load("pt_core_news_md")


# =========================
# LIMPEZA
# =========================
def limpar(texto: str) -> str:
    texto = str(texto).lower()
    texto = texto.replace("não", "nao")
    texto = unidecode(texto)
    texto = re.sub(r'[^a-z0-9\s]', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto


# =========================
# EMBEDDINGS SPACY
# =========================
def spacy_vector(texto):
    return nlp(texto).vector


# =========================
# DATASET
# =========================
df = pd.read_csv("dataset_2000_tickets_full.csv")

df = df.drop_duplicates(subset=["Titulo", "Descricao", "Categoria"])

df["Titulo"] = df["Titulo"].fillna("")
df["Descricao"] = df["Descricao"].fillna("")
df["Categoria"] = df["Categoria"].fillna("")

df["Texto"] = (df["Titulo"] + " " + df["Descricao"]).apply(limpar)

df = df[(df["Texto"].str.strip() != "") & (df["Categoria"].str.strip() != "")]


# =========================
# DATASET EXTRA
# =========================
dataset_extra = [
("pc nao liga", "hardware"),
("erro smtp", "email"),
("malware detectado", "seguranca"),
("impressora nao imprime", "impressora"),
("wifi nao conecta", "redes"),
("programa nao abre", "software"),
("erro 500", "servidor"),
("nao consigo login", "acesso"),

("internet lenta e instavel", "redes"),
("pc trava quando abre programa", "hardware"),
("email nao envia mas internet funciona", "email"),
("erro estranho no sistema", "outros"),
("problema geral sem descricao clara", "outros"),
("pc ta estranho e nao sei explicar", "outros"),
]

df_extra = pd.DataFrame(dataset_extra, columns=["Texto", "Categoria"])
df_extra["Texto"] = df_extra["Texto"].apply(limpar)

df = pd.concat([df, df_extra], ignore_index=True)


# =========================
# 🔥 CORREÇÃO CRÍTICA (EVITA SEU ERRO)
# =========================
print("\n📊 DISTRIBUIÇÃO ANTES DO FILTRO:")
print(df["Categoria"].value_counts())

counts = df["Categoria"].value_counts()

# remove classes com menos de 2 exemplos
valid_classes = counts[counts >= 2].index
df = df[df["Categoria"].isin(valid_classes)]

print("\n📊 DISTRIBUIÇÃO APÓS FILTRO:")
print(df["Categoria"].value_counts())


# =========================
# SPLIT SEGURO
# =========================
X = df["Texto"]
y = df["Categoria"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# =========================
# TF-IDF
# =========================
tfidf = TfidfVectorizer(
    stop_words=stopwords_pt,
    ngram_range=(1,2),
    max_df=0.85,
    min_df=2,
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)


# =========================
# SPACY EMBEDDINGS
# =========================
X_train_spacy = np.array([spacy_vector(t) for t in X_train])
X_test_spacy = np.array([spacy_vector(t) for t in X_test])


# =========================
# VETOR HÍBRIDO
# =========================
X_train_final = np.hstack([X_train_tfidf.toarray(), X_train_spacy])
X_test_final = np.hstack([X_test_tfidf.toarray(), X_test_spacy])


# =========================
# MODELO
# =========================
model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced"
)

model.fit(X_train_final, y_train)


# =========================
# REGRAS HÍBRIDAS
# =========================
def regra_segura(texto):
    t = texto.lower()

    if any(x in t for x in ["malware", "virus", "phishing", "hack"]):
        return "seguranca"

    if any(x in t for x in ["impressora", "spooler", "impressao"]):
        return "impressora"

    if any(x in t for x in ["erro 500", "erro 503", "timeout"]):
        return "servidor"

    return None


# =========================
# PREVISÃO
# =========================
def prever(texto):
    texto_limpo = limpar(texto)

    regra = regra_segura(texto_limpo)
    if regra:
        return regra

    x_tfidf = tfidf.transform([texto_limpo]).toarray()
    x_spacy = np.array([spacy_vector(texto_limpo)])

    x_final = np.hstack([x_tfidf, x_spacy])

    return model.predict(x_final)[0]


# =========================
# AVALIAÇÃO
# =========================
y_pred = model.predict(X_test_final)

print("\n📊 CLASSIFICATION REPORT")
print(classification_report(y_test, y_pred, zero_division=0))

print("\n📊 MATRIZ DE CONFUSÃO")
print(confusion_matrix(y_test, y_pred))


# =========================
# TESTES
# =========================
testes_reais = {
"hardware": ["pc nao liga", "hd queimado", "monitor sem sinal"],
"email": ["nao recebo email"],
"seguranca": ["malware detectado"],
"impressora": ["impressora nao imprime"],
"redes": ["wifi nao conecta"],
"software": ["programa nao abre"],
"servidor": ["erro 500"],
"acesso": ["nao consigo login"],
"outros": ["minha internet tá estranha mas não sei o que é"]
}

print("\n🔎 TESTE FINAL\n")

total = 0
acertos = 0

for cat, frases in testes_reais.items():
    print(f"\n===== {cat.upper()} =====")

    for f in frases:
        p = prever(f)
        total += 1

        if p == cat:
            acertos += 1
            print(f"✅ {f}")
        else:
            print(f"❌ {f} -> previsto: {p}")


print(f"\n📊 ACURÁCIA FINAL: {acertos}/{total} = {acertos/total*100:.2f}%")


# =========================
# SALVAMENTO
# =========================
joblib.dump({
    "model": model,
    "tfidf": tfidf,
    "limpar": limpar,
    "prever": prever
}, "classificador_tickets_spacy_tfidf.pkl")

print("\n💾 MODELO FINAL ESTÁVEL (TF-IDF + SPACY)")